# `crossref`, the parallel witness edge, and what to do with it

The corpus publishes several Palestinian Targum manuscripts of the same five books. Where two of
them translate one verse, that verse exists twice, and the pair of readings is the unit of a
critical comparison. `crossref` is the edge that joins them.

    E.crossref.f(verse)  ->  ((other_verse, 'witness' | 'view'), ...)

**Two verses are parallel iff they carry the same `(book, chapter, verse)`.** That is the whole
definition. The relation is *given* by the data, not inferred: there is no similarity threshold,
no choice of method and no weight to interpret. This is the one way it differs from ETCBC's
`parallels`, where three methods ship (`crossref`, `crossrefSET`, `crossrefLCS`) precisely
because the links there are computed and a reader has to decide how much similarity counts.

### The label is not decoration

| `relation` | meaning | ordered pairs |
|---|---|---|
| `witness` | two **manuscripts** translate this verse. **This is the parallel.** | 76,888 |
| `view` | one manuscript under two **views**. `neofiti_base` and `neofiti_full` are the same codex, Neofiti 1. **Not a parallel.** | 11,654 |
| `same_witness` | one manuscript **and** one view type, holding more than one verse record at this address. **Neither.** | 20 |

Collapse `witness` and `view` and *"how many witnesses attest this verse"* is wrong by one on all
5,827 Neofiti addresses. The `manuscript` feature on the `text` tier is what lets you tell them
apart, and it was published for this reason: `text` names the manuscript everywhere **except**
Neofiti, where two `text` values are one codex. 47 names, 46 manuscripts, one collapse.

`same_witness` is small and easy to forget, which is why it has a label rather than an exclusion.
The relation is **total** over the address, in that every pair of `verse` nodes sharing
`(book, chapter, verse)` is joined, and three Genizah addresses hold more than one verse record
of one fragment: `_dd` at Deuteronomy 24:6 (2 records), `_ff` at Exodus 15:12 (4) and 17:12 (3).
Those pairs are not parallels and are not views. Filter on `relation == "witness"` and they cost
you nothing; ignore the label and you will count a fragment as its own witness.

### Where the relation lives

| artefact | keyed on | use it for |
|---|---|---|
| `tf/0.1/crossref.tf` | Text-Fabric **node numbers** | analysis, which is this notebook |
| `data/crossref/crossref_edges.csv` | `(manuscript, view_type)` at the canonical address | citation, and anything that must survive a recompile |
| `crossref_verses` / `crossref_edges` / `crossref_degree` in PostgreSQL | the same | SQL-side work |

Node numbers move on every recompile and the address does not, so the CSV is the citable
statement and the edge is the convenient one. The verification mode of `programs/crossref_module.py`
is where the two are compared, and it is the only place they should be.

### What this notebook reads with

**context-fabric**, like every notebook in `programs/`. There is no advanced application layer, so no
`A.pretty` and no `A.table`, and the display helpers below print instead. It addresses the corpus by *path*
rather than by `org/repo`, which is what lets this file run unchanged in the public
`ETCBC/targum` repository, where there is no `.env` and nothing to resolve.

In [1]:
# Loads the corpus and builds the two lookups every example below uses: the canonical address of
# each verse node, and the (manuscript, view) its text node carries.
#
# Both are plain dicts over the 19,595 verse nodes, which is small enough that building them
# eagerly is cheaper than resolving `L.u` inside each example.

from collections import Counter, defaultdict
from pathlib import Path

import yaml

import cfabric

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/xml.yml").exists())
TF_VERSION = yaml.safe_load((REPO_ROOT / "app/config.yaml").read_text())["provenanceSpec"]["version"]
TF_DIR = REPO_ROOT / "tf" / TF_VERSION

# The silence level is passed to both calls, because the loader takes its own. Without it the
# library logs one line per feature as it loads, which is longer than most of the output and
# conveys nothing that the inventory does not, and it tells a
# reader nothing the feature inventory does not. The accepted levels are "verbose", "auto",
# "terse" and "deep"; it has to be passed twice because `loadAll` takes its own.
CF = cfabric.Fabric(locations=str(REPO_ROOT / "tf"), modules=TF_VERSION, silent="deep")
api = CF.loadAll(silent="deep")
F, E, L, T, S = api.F, api.E, api.L, api.T, api.S
Fs, Es = api.Fs, api.Es

assert hasattr(E, "crossref"), "this dataset carries no `crossref` edge, so recompile the corpus"


def verse_number(v):
    """`verse` as an int. The eleven unnumbered verses publish 0, or 0 and -2 in one chapter.

    -1 is skipped deliberately: it is context-fabric's reserved missing-value sentinel for an
    int feature, so a published -1 would arrive here as None.
    """
    n = F.verse.v(v)
    return 0 if n is None else int(n)


ADDRESS, WITNESS = {}, {}
for v in F.otype.s("verse"):
    t = L.u(v, otype="text")[0]
    ADDRESS[v] = (F.book.v(L.u(v, otype="book")[0]),
                  F.chapter.v(L.u(v, otype="chapter")[0]),
                  verse_number(v))
    WITNESS[v] = (F.manuscript.v(t), F.view.v(t))

AT = defaultdict(list)
for v, a in ADDRESS.items():
    AT[a].append(v)


def words(v):
    """The vocalised surface of a verse, space-joined, which is the cheapest readable form."""
    return " ".join(F.voc.v(w) or "" for w in L.d(v, otype="word"))


VIEWS_PER_MS = defaultdict(set)
for _ms, _view in WITNESS.values():
    VIEWS_PER_MS[_ms].add(_view)


def label(v):
    """How every table below names a witness: the manuscript, and the view only where it
    distinguishes two readings of one manuscript, which here is Neofiti 1 and nothing else.
    45 of the 46 text nodes carry `full_text`, so printing it everywhere would be noise."""
    ms, view = WITNESS[v]
    return f"{ms} [{view}]" if len(VIEWS_PER_MS[ms]) > 1 else ms


print(f"corpus      {TF_DIR}")
print(f"verse nodes {len(ADDRESS):,}")
print(f"addresses   {len(AT):,}")
print(f"manuscripts {len({m for m, _ in WITNESS.values()}):,}")
print(f"text nodes  {len(list(F.otype.s('text'))):,}")

corpus      /Users/sfir1/dissertation/tf/0.1
verse nodes 19,595
addresses   5,866
manuscripts 45
text nodes  46


## 1. The one-line use: *who else has this verse?*

This is what the edge is for. Take a verse, read its edges, and you have every other reading of
it in the corpus, already labelled with whether it is a different manuscript or the same one read
a different way.

Note the two filters. `relation == 'witness'` is the parallel; counting *distinct manuscripts*
rather than *edges* matters because a manuscript can contribute two verse nodes to one address
when its verses interleave, which accounts for six such nodes.

In [2]:
RELATIONS = ("witness", "view", "same_witness")


def witnesses(v, relation="witness"):
    """The other readings of this verse. `relation=None` for all three kinds."""
    return [(o, r) for o, r in (E.crossref.f(v) or ())
            if relation is None or r == relation]


start = T.nodeFromSection(("Genesis", 44, 18))
print(f"{label(start)}  {ADDRESS[start][0]} {ADDRESS[start][1]}:{ADDRESS[start][2]}")
print(f"  {words(start)[:96]}...\n")

parallel = witnesses(start)
print(f"  {len(parallel)} witness edge(s) to "
      f"{len({WITNESS[o][0] for o, _ in parallel})} distinct manuscript(s)\n")
for o, r in sorted(parallel, key=lambda p: label(p[0])):
    print(f"  {r:<8} {label(o):<34} {len(L.d(o, otype='word')):>4} words")

neofiti_1 [full_text]  Genesis 44:18
  W QRB LWWTJH JHWDH Z<P B MLJN W MDKDK B LJCN> NHM K >RJH W >MR BB<W MJNK RBWNJ JMLL K<N <BDK PTG...

  7 witness edge(s) to 7 distinct manuscript(s)

  witness  cairo_genizah_targum_fragment_m      82 words
  witness  cairo_genizah_targum_fragment_r     132 words
  witness  cairo_genizah_targum_fragment_rr     83 words
  witness  cairo_genizah_targum_fragment_z     195 words
  witness  fragment_targum_p                   242 words
  witness  fragment_targum_v                   297 words
  witness  pseudo_jonathan                      49 words


## 2. A critical apparatus for one address

Walk the edges, print every reading side by side. This is the skeleton of the apparatus: the edge
supplies the population and the corpus supplies the text, so there is no alignment step and
nothing to tune.

`sorted(..., key=label)` is deliberate, giving manuscript order rather than node order, so the same address
prints the same way whoever generated it.

In [3]:
def apparatus(book, chapter, verse, width=88):
    """Every reading of one address, in manuscript order.

    `book` is a string while `chapter` and `verse` are integers, both being declared as integer
    features, which is what makes a numeric sort of them correct.
    """
    here = AT.get((book, int(chapter), int(verse)))
    if not here:
        print(f"no witness at {book} {chapter}:{verse}")
        return []
    print(f"{book} {chapter}:{verse}  |  {len(here)} verse node(s), "
          f"{len({WITNESS[v][0] for v in here})} manuscript(s)\n")
    for v in sorted(here, key=label):
        text = words(v)
        print(f"  {label(v):<34} {len(L.d(v, otype='word')):>4}w  "
              f"{text[:width]}{'...' if len(text) > width else ''}")
    return here


_ = apparatus("Genesis", 44, 18)

Genesis 44:18  |  9 verse node(s), 8 manuscript(s)

  cairo_genizah_targum_fragment_m      82w  W QARJB LWATJH JHUWD@H WD.@> ... ... W>AMAR  BB@<W RJBWNJ H> B ZMN> Q DMJ T> KAD >ATJN> ...
  cairo_genizah_targum_fragment_r     132w  >APR;JCT@> MIJN.EN@> WIJHABT;JH B B; JT MAVR@> L@> >IJD.KAR L@K W L@> >IJCTMOWDA< L@K M@...
  cairo_genizah_targum_fragment_rr     83w  ... ...  ... W>XS NH BPIJLW G ... ... W>JT L;JH X WLQ ... XAMWOR W.BN WJ ... ... <J LJCR...
  cairo_genizah_targum_fragment_z     195w  W QRJB LWTJH JHWDH HJK >RJ> W >MR BB <W MNK RBWNJ JMLL K<N <BDK PTGM B MCM<J D  RBWNJ W ...
  fragment_targum_p                   242w  W QRJB LWWTJH JHWDH W >MR BB<W MJNK RBWNJ JMLL K<N <BDK PTGM B MCM<JH D RBWNJ W L> JTQP ...
  fragment_targum_v                   297w  W QRB LWWTH JHWDH HK >RJ> W >MR BB<W MJNK RJBWNJ JMLL K<N <BDK PJTGM B MJCM<J D RJBWNJ W...
  neofiti_1 [base_raw]                459w  W QRB LWWTJH JHWDH Z<P B MLJN W MDKDK B LJCN> NHM K >RJH W >MR BB<W MJNK RBWNJ JMLL K<N 

## 3. How deep is the attestation, address by address?

The first question to ask of a parallel corpus is where the comparison is actually possible. Most
addresses have two manuscripts; a handful have eight. Those are where a critical edition has
something to argue about.

Count **manuscripts**, not verse nodes: Neofiti's two views are one codex, and interleaved verses
add nodes without adding witnesses.

In [4]:
depth = Counter(len({WITNESS[v][0] for v in nodes}) for nodes in AT.values())

print(f"  {'manuscripts':>12}  {'addresses':>10}  {'% of corpus':>11}")
print("  " + "-" * 37)
for n in sorted(depth, reverse=True):
    print(f"  {n:>12}  {depth[n]:>10,}  {100.0 * depth[n] / len(AT):>10.2f}%")
print("  " + "-" * 37)
print(f"  {'total':>12}  {len(AT):>10,}")
print(f"\n  addresses with a parallel: {len(AT) - depth[1]:,} of {len(AT):,}")
print(f"  addresses attested once:   {depth[1]:,}  (no witness edge at all)")

deepest = sorted(AT.items(), key=lambda kv: -len({WITNESS[v][0] for v in kv[1]}))[:8]
print("\n  the most widely attested addresses")
for addr, nodes in deepest:
    print(f"    {addr[0]:<12} {addr[1]:>3}:{addr[2]:<4} "
          f"{len({WITNESS[v][0] for v in nodes})} manuscripts  "
          f"{', '.join(sorted({WITNESS[v][0] for v in nodes}))[:72]}")

   manuscripts   addresses  % of corpus
  -------------------------------------
             8           3        0.05%
             7          14        0.24%
             6          20        0.34%
             5          86        1.47%
             4         433        7.38%
             3         777       13.25%
             2       4,494       76.61%
             1          39        0.66%
  -------------------------------------
         total       5,866

  addresses with a parallel: 5,827 of 5,866
  addresses attested once:   39  (no witness edge at all)

  the most widely attested addresses
    Genesis       44:18   8 manuscripts  cairo_genizah_targum_fragment_m, cairo_genizah_targum_fragment_r, cairo_
    Exodus        12:2    8 manuscripts  cairo_genizah_targum_fragment_aa, cairo_genizah_targum_fragment_gg, cair
    Exodus        19:3    8 manuscripts  cairo_genizah_targum_fragment_f, cairo_genizah_targum_fragment_j, cairo_
    Genesis        4:8    7 manuscripts  cairo_gen

## 4. Where do the witnesses disagree? The cheap signal first

Word count. It needs no alignment, no transliteration comparison and no judgement, and it finds
the addresses where one witness is expansive and another terse, which in the Palestinian Targum
tradition is usually an *aggadic expansion*, the thing the tradition is interesting for.

Restricted to addresses with three or more manuscripts, so that a spread is a disagreement among
witnesses rather than the difference between two copies.

In [5]:
spread = []
for addr, nodes in AT.items():
    by_ms = defaultdict(int)
    for v in nodes:
        by_ms[WITNESS[v][0]] += len(L.d(v, otype="word"))
    if len(by_ms) < 3:
        continue
    lo, hi = min(by_ms.values()), max(by_ms.values())
    spread.append((hi - lo, addr, lo, hi, by_ms))
spread.sort(reverse=True)

print(f"  {len(spread):,} addresses carry three or more manuscripts\n")
print(f"  {'address':<22} {'min':>5} {'max':>5} {'spread':>7}   widest / narrowest")
print("  " + "-" * 94)
for d, addr, lo, hi, by_ms in spread[:10]:
    widest = max(by_ms, key=by_ms.get)
    narrowest = min(by_ms, key=by_ms.get)
    print(f"  {addr[0]+' '+str(addr[1])+':'+str(addr[2]):<22} {lo:>5} {hi:>5} {d:>7}   "
          f"{widest} ({hi}) / {narrowest} ({lo})")

  1,333 addresses carry three or more manuscripts

  address                  min   max  spread   widest / narrowest
  ----------------------------------------------------------------------------------------------
  Genesis 44:18             49   924     875   neofiti_1 (924) / pseudo_jonathan (49)
  Genesis 38:25             97   659     562   neofiti_1 (659) / cairo_genizah_targum_fragment_x (97)
  Exodus 12:42              12   507     495   neofiti_1 (507) / cairo_genizah_targum_fragment_aa (12)
  Exodus 12:2               15   496     481   fragment_targum_p (496) / cairo_genizah_targum_fragment_aa (15)
  Leviticus 22:27           40   495     455   neofiti_1 (495) / cairo_genizah_targum_fragment_ff (40)
  Exodus 14:29              16   413     397   cairo_genizah_targum_fragment_pp (413) / pseudo_jonathan (16)
  Numbers 34:15             12   353     341   neofiti_1 (353) / pseudo_jonathan (12)
  Deuteronomy 32:1         116   444     328   neofiti_1 (444) / pseudo_jonathan (116)

## 5. Lexical divergence: what does one witness say that another does not?

One step past word count, and still mechanical. Take an address, take two manuscripts, and
difference their lexeme sets. `lex0` is the lexeme without its homograph markers, which is the
right granularity for "is this word present at all".

This is the shape of a real apparatus entry: not "these verses differ" but *which lemma* one
witness has and the other has not.

In [6]:
def lexemes(v):
    """The `lex0` set of a verse node, the lexeme stripped of its homograph markers."""
    return {F.lex0.v(w) for w in L.d(v, otype="word") if F.lex0.v(w)}


def diverge(v, other):
    """Lexemes in one reading and not the other, both ways."""
    a, b = lexemes(v), lexemes(other)
    return sorted(a - b), sorted(b - a)


_, widest_addr, _, _, _ = spread[0]
nodes = sorted(AT[widest_addr], key=label)
base = max(nodes, key=lambda v: len(L.d(v, otype="word")))

print(f"{widest_addr[0]} {widest_addr[1]}:{widest_addr[2]}   "
      f"base = {label(base)} ({len(lexemes(base))} lexemes)\n")
for o, r in sorted(witnesses(base, relation=None), key=lambda p: label(p[0])):
    only_base, only_other = diverge(base, o)
    print(f"  {r:<8} {label(o):<30} {len(lexemes(o)):>3} lexemes  "
          f"+{len(only_base):>3} / -{len(only_other):>3}")
    if only_other:
        print(f"           only in {WITNESS[o][0]}: {' '.join(only_other[:14])}")

Genesis 44:18   base = neofiti_1 [full_text] (132 lexemes)

  witness  cairo_genizah_targum_fragment_m  45 lexemes  + 91 / -  4
           only in cairo_genizah_targum_fragment_m: KD T<R TXL XJ
  witness  cairo_genizah_targum_fragment_r  80 lexemes  + 82 / - 30
           only in cairo_genizah_targum_fragment_r: <LXD <SR >CTMWD< >LH BDJL BJ BKN BNJMJN BR CNH DKR JCR>L JD JFR>L
  witness  cairo_genizah_targum_fragment_rr   0 lexemes  +132 / -  0
  witness  cairo_genizah_targum_fragment_z  86 lexemes  + 58 / - 12
           only in cairo_genizah_targum_fragment_z: <LXD <WD >DNJ >L> BNJMJN DXJL HJK JD KPR MCM< NXT XVJ
  witness  fragment_targum_p              106 lexemes  + 41 / - 15
           only in fragment_targum_p: >NC BRM DKMT DKR HCT> MCM< MNWD> PLJGH Q> QVWL QWMJ RWXYN TXL XJB
  witness  fragment_targum_v              111 lexemes  + 32 / - 11
           only in fragment_targum_v: <LXD BCLWWH BNJMJN CQR CWLVN HJK JD JTB MRXQ NXT YDW
  view     neofiti_1 [base_raw]           129 le

## 6. The `view` edges on their own: what an editorial filter removes

`view` edges are *not* parallels, and that is exactly why they are worth a section. They join one
codex to itself under two editorial filters. `neofiti_base_raw` keeps only unmarked segments,
`neofiti_full` places every segment side by side. So the difference between the two ends of a
`view` edge is precisely **the apparatus of Neofiti 1**: the variants, additions and interlinear
material that the base text suppresses.

No other pair in the corpus has this property, which is the reason the relation is labelled
rather than left bare.

In [7]:
view_pairs = {}
for v in F.otype.s("verse"):
    for o, r in E.crossref.f(v) or ():
        if r == "view" and (o, v) not in view_pairs:
            view_pairs[(v, o)] = ADDRESS[v]

print(f"  {len(view_pairs):,} unordered `view` pairs over "
      f"{len(set(view_pairs.values())):,} addresses\n")

added = []
for (a, b), addr in view_pairs.items():
    base, full = (a, b) if WITNESS[a][1] == "base_raw" else (b, a)
    gain = len(L.d(full, otype="word")) - len(L.d(base, otype="word"))
    added.append((gain, addr, base, full))
added.sort(reverse=True)

total_base = sum(len(L.d(b, otype="word")) for _, _, b, _ in added)
total_full = sum(len(L.d(f, otype="word")) for _, _, _, f in added)
print(f"  Neofiti base  {total_base:>8,} words")
print(f"  Neofiti full  {total_full:>8,} words")
print(f"  the apparatus {total_full - total_base:>8,} words "
      f"({100.0 * (total_full - total_base) / total_base:.1f}% of the base text)\n")
print("  the verses where the filter removes the most")
for gain, addr, base, full in added[:8]:
    print(f"    {addr[0]:<12} {addr[1]:>3}:{addr[2]:<4} "
          f"base {len(L.d(base, otype='word')):>4}w -> full {len(L.d(full, otype='word')):>4}w "
          f"(+{gain})")

  5,827 unordered `view` pairs over 5,827 addresses

  Neofiti base   134,152 words
  Neofiti full   158,026 words
  the apparatus   23,874 words (17.8% of the base text)

  the verses where the filter removes the most
    Exodus        12:42   base  232w -> full  275w (+43)
    Deuteronomy   14:21   base   26w -> full   66w (+40)
    Genesis       16:5    base  140w -> full  178w (+38)
    Numbers       11:31   base   30w -> full   66w (+36)
    Deuteronomy   17:19   base    4w -> full   40w (+36)
    Numbers       21:6    base  109w -> full  144w (+35)
    Genesis       18:25   base   16w -> full   51w (+35)
    Deuteronomy    9:23   base   16w -> full   49w (+33)


## 7. A pairwise overlap matrix: which manuscripts can be compared at all?

Before comparing two manuscripts you need to know how much they share. Count addresses where
both are present. It is derived from the edge, so it needs no coverage table and cannot disagree with
the relation it summarises.

The diagonal is each manuscript's own address count, which is also the honest denominator for
any percentage taken off a row.

In [8]:
held = defaultdict(set)
for v, addr in ADDRESS.items():
    held[WITNESS[v][0]].add(addr)

order = sorted(held, key=lambda m: -len(held[m]))
top = order[:9]
w = max(len(m) for m in top)

print("  shared addresses, pairwise (diagonal = the manuscript's own total)\n")
print("  " + " " * w + "  " + "  ".join(f"{m[:7]:>7}" for m in top))
for a in top:
    row = []
    for b in top:
        row.append(f"{len(held[a] & held[b]):>7,}")
    print(f"  {a:<{w}}  " + "  ".join(row))

print(f"\n  {len(order)} manuscripts in total; "
      f"{sum(1 for m in order if len(held[m]) < 20)} hold fewer than 20 addresses each")

  shared addresses, pairwise (diagonal = the manuscript's own total)

                                    neofiti  pseudo_  fragmen  fragmen  cairo_g  cairo_g  cairo_g  cairo_g  cairo_g
  neofiti_1                           5,844    5,822      913      591       57       44       42       41       40
  pseudo_jonathan                     5,822    5,831      912      589       56       43       42       41       40
  fragment_targum_v                     913      912      920      454        3       26        6        5        5
  fragment_targum_p                     591      589      454      594        2       26        3        5        5
  cairo_genizah_targum_fragment_e        57       56        3        2       57        0        0        0        0
  cairo_genizah_targum_fragment_f        44       43       26       26        0       44        0        0        0
  cairo_genizah_targum_fragment_aa       42       42        6        3        0        0       42        0        0
  

## 8. Driving the edge from a search

`S.search` finds the population and the edge expands it. Here: every verse whose text carries a
segment marked `variant`, then the other witnesses to those same addresses, which is the
practical form of *"where this manuscript records a variant, what do the others read?"*

This is the composition worth remembering. The edge does not replace search; it turns any result
set into a set of comparisons.

In [9]:
query = """
verse
  segment type=variant
"""
found = list(S.search(query))
hits = sorted({r[0] for r in found})
print(f"  {len(hits):,} verse nodes carry a `variant` segment\n")

compared = [v for v in hits if witnesses(v)]
print(f"  of those, {len(compared):,} have at least one other MANUSCRIPT at the same address")
print(f"  and {len(hits) - len(compared):,} do not, so their variant cannot be checked "
      f"against another witness\n")

by_ms = Counter(WITNESS[v][0] for v in compared)
print("  comparable variant-bearing verses, by manuscript")
for ms, n in by_ms.most_common(8):
    print(f"    {ms:<24} {n:>6,}")

  4,655 verse nodes carry a `variant` segment

  of those, 4,648 have at least one other MANUSCRIPT at the same address
  and 7 do not, so their variant cannot be checked against another witness

  comparable variant-bearing verses, by manuscript
    neofiti_1                 4,618
    pseudo_jonathan              20
    fragment_targum_p             7
    fragment_targum_v             3


## 9. Going back to an address-keyed form

Everything above is keyed on node numbers, which move on every recompile. When a result has to
*outlive the build*, such as a citation, a table in a paper or a file someone else reads, re-key it on
`(book, chapter, verse, manuscript, view)` first. That is the form
`data/crossref/crossref_edges.csv` already ships, and this is how to get there from the edge.

The counts differ by construction and that is not an error. The edge carries 88,562 ordered pairs
and the CSV 88,458: 84 extra `witness` pairs, because the CSV collapses a manuscript contributing
two verse nodes to one address and a node-keyed relation cannot, plus the 20 `same_witness` pairs
the CSV has no row for at all. Collapsed onto `(manuscript, view)` with `same_witness` set aside,
the two agree exactly, which is what the verification mode of `programs/crossref_module.py` asserts.

In [10]:
import csv, io

rows, same_witness = [], []
for v in F.otype.s("verse"):
    for o, r in E.crossref.f(v) or ():
        addr = ADDRESS[v]
        row = (addr[0], addr[1], addr[2], *WITNESS[v], *WITNESS[o], r)
        (same_witness if r == "same_witness" else rows).append(row)

collapsed = sorted(set(rows))
print(f"  ordered node pairs           {len(rows) + len(same_witness):>8,}")
print(f"    of them `same_witness`     {len(same_witness):>8,}  "
      f"(set aside, because the CSV has no row for these)")
print(f"  address-keyed, distinct      {len(collapsed):>8,}  "
      f"<- compare with crossref_edges.csv")
print(f"    collapsed away             {len(rows) - len(collapsed):>8,}  "
      f"(a manuscript with two verse nodes at one address)\n")

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerow(["book", "chapter", "verse", "from_manuscript", "from_view",
                 "to_manuscript", "to_view", "relation"])
writer.writerows(collapsed[:4])
print(buf.getvalue())
print("  write it with:  Path('crossref_from_tf.csv').write_text(buf.getvalue())")

  ordered node pairs             49,404
    of them `same_witness`           20  (set aside, because the CSV has no row for these)
  address-keyed, distinct        49,312  <- compare with crossref_edges.csv
    collapsed away                   72  (a manuscript with two verse nodes at one address)

book,chapter,verse,from_manuscript,from_view,to_manuscript,to_view,relation
Deuteronomy,1,1,cairo_genizah_targum_fragment_br,full_text,fragment_targum_p,full_text,witness
Deuteronomy,1,1,cairo_genizah_targum_fragment_br,full_text,fragment_targum_v,full_text,witness
Deuteronomy,1,1,cairo_genizah_targum_fragment_br,full_text,neofiti_1,base_raw,witness
Deuteronomy,1,1,cairo_genizah_targum_fragment_br,full_text,neofiti_1,full_text,witness

  write it with:  Path('crossref_from_tf.csv').write_text(buf.getvalue())


## What to reach for

| question | the call |
|---|---|
| other readings of this verse | `E.crossref.f(v)` |
| only the parallels | `[o for o, r in E.crossref.f(v) if r == 'witness']` |
| how many witnesses attest it | `len({F.manuscript.v(L.u(o, otype='text')[0]) for o, r in ... if r == 'witness'})` |
| Neofiti's apparatus for this verse | the `view` edge, §6 |
| the other records of *this* witness | the `same_witness` edge, 20 ordered pairs over 3 addresses |
| is this verse comparable at all | `bool(witnesses(v))`, where 39 addresses answer no |
| the same relation, surviving a recompile | `data/crossref/crossref_edges.csv`, or §9 |

### Three traps

1. **Counting edges where you mean manuscripts.** A manuscript can contribute two verse nodes to
   one address. Deduplicate on `manuscript` whenever the answer is "how many witnesses".
2. **Treating a `view` edge as a parallel.** It is one codex, twice. Filter on `relation`, always.
3. **Forgetting `same_witness` exists.** It is 20 ordered pairs out of 88,562, so it will not
   show up in a spot check, and it is the one value where both ends are the same witness.

### Not in 0.1

No alignment *within* the verse. Word-to-word correspondence between two readings is not
published, and nothing here infers it. §4 and §5 are the mechanical signals available instead,
and they are deliberately crude: they find where to look, not what to conclude.